<a href="https://colab.research.google.com/github/rasushi/AskDocs/blob/main/04_evaluation_and_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

sushmithaveigas_fundus_avseg_path = kagglehub.dataset_download('sushmithaveigas/fundus-avseg')
sushmithaveigas_driave_path = kagglehub.dataset_download('sushmithaveigas/driave')
sushmithaveigas_dataprep_op_avseg_path = kagglehub.dataset_download('sushmithaveigas/dataprep-op-avseg')
sushmithaveigas_unet_efficientnet_b4_path = kagglehub.dataset_download('sushmithaveigas/unet-efficientnet-b4')
sushmithaveigas_unet_resnet50_path = kagglehub.dataset_download('sushmithaveigas/unet-resnet50')
sushmithaveigas_unet_resnet34_path = kagglehub.dataset_download('sushmithaveigas/unet-resnet34')

print('Data source import complete.')


In [ ]:
!pip install segmentation-models-pytorch albumentations opencv-python-headless matplotlib --no-deps

In [ ]:
# ============================================================================
# AI-BASED ARTERY-VEIN SEGMENTATION - EVALUATION & ANALYSIS
# ============================================================================
# This notebook contains:
# - Phase 1: Model Evaluation on Test Set
# - Phase 2: Qualitative Visual Analysis
# - Phase 3: Efficiency & Speed Analysis
# - Phase 4: Statistical Validation
# - Phase 5: Best Model Selection
# ============================================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
from collections import defaultdict
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{'='*70}")
print(f"EVALUATION ENVIRONMENT".center(70))
print(f"{'='*70}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"{'='*70}\n")

# Create output directory for all results
OUTPUT_DIR = '/kaggle/working/evaluation_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/tables", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/predictions", exist_ok=True)

print(f"✅ Output directory created: {OUTPUT_DIR}\n")

In [ ]:
# ============================================================================
# SECTION 1: DATASET & TRANSFORMS SETUP
# ============================================================================

class RetinalAVSegmentationDataset(Dataset):
    """Dataset class for loading retinal images and masks"""
    def __init__(self, json_path, transforms=None, image_size=512):
        with open(json_path, 'r') as f:
            self.data = json.load(f)

        self.transforms = transforms
        self.image_size = image_size

        print(f"✓ Loaded {len(self.data)} samples from {json_path}")

    def __len__(self):
        return len(self.data)

    def load_image(self, image_path):
        if image_path.endswith('.tif'):
            image = Image.open(image_path)
            image = np.array(image)
        else:
            image = cv2.imread(image_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

    def load_mask(self, mask_path):
        mask = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        class_mask = self.convert_to_class_mask(mask_rgb)
        return class_mask

    def convert_to_class_mask(self, color_mask):
        h, w, _ = color_mask.shape
        class_mask = np.zeros((h, w), dtype=np.int64)

        # Red (artery) - high R, low G, low B
        red_mask = (color_mask[:, :, 0] > 200) & \
                   (color_mask[:, :, 1] < 50) & \
                   (color_mask[:, :, 2] < 50)

        # Blue (vein) - low R, low G, high B
        blue_mask = (color_mask[:, :, 2] > 200) & \
                    (color_mask[:, :, 0] < 50) & \
                    (color_mask[:, :, 1] < 50)

        class_mask[red_mask] = 1
        class_mask[blue_mask] = 2

        return class_mask

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = self.load_image(sample['image'])
        mask = self.load_mask(sample['mask'])

        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']

        return {
            'image': image,
            'mask': mask,
            'id': sample['id'],
            'dataset': sample['dataset']
        }

def get_validation_transforms(image_size=512):
    """Validation transforms (no augmentation)"""
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
        ToTensorV2(),
    ])

# Load test dataset
print(f"\n{'='*70}")
print("LOADING TEST DATASET".center(70))
print(f"{'='*70}\n")

TEST_JSON = '/kaggle/input/dataprep-op-avseg/processed_data/test_files.json'
IMAGE_SIZE = 512
BATCH_SIZE = 8

test_dataset = RetinalAVSegmentationDataset(
    TEST_JSON,
    transforms=get_validation_transforms(IMAGE_SIZE),
    image_size=IMAGE_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✅ Test DataLoader created: {len(test_loader)} batches\n")

In [ ]:
# ============================================================================
# SECTION 2: MODEL LOADING & CONFIGURATION
# ============================================================================

# Model configurations
MODEL_CONFIGS = {
    'model1': {
        'name': 'UNet-ResNet34',
        'architecture': 'UNet',
        'encoder': 'resnet34',
        'checkpoint': '/kaggle/input/unet-resnet34/model1_outputs/model1_best.pth',
        'color': '#FF6B6B',
        'marker': 'o'
    },
    'model2': {
        'name': 'UNet-ResNet50',
        'architecture': 'UNet',
        'encoder': 'resnet50',
        'checkpoint': '/kaggle/input/unet-resnet50/model2_outputs/model2_best.pth',
        'color': '#4ECDC4',
        'marker': 's'
    },
    'model3': {
        'name': 'UNet++-EfficientNet-B4',
        'architecture': 'UNetPlusPlus',
        'encoder': 'efficientnet-b4',
        'checkpoint': '/kaggle/input/unet-efficientnet-b4/model3_outputs/model3_best.pth',
        'color': '#95E1D3',
        'marker': '^'
    }
}

def load_model(config):
    """Load a trained model from checkpoint"""
    print(f"Loading {config['name']}...")

    # Create model
    if config['architecture'] == 'UNet':
        model = smp.Unet(
            encoder_name=config['encoder'],
            encoder_weights=None,
            in_channels=3,
            classes=3,
        )
    else:  # UNetPlusPlus
        model = smp.UnetPlusPlus(
            encoder_name=config['encoder'],
            encoder_weights=None,
            in_channels=3,
            classes=3,
        )

    # Load checkpoint
    checkpoint = torch.load(config['checkpoint'], map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()

    # Get model info
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"  ✓ Architecture: {config['architecture']}")
    print(f"  ✓ Encoder: {config['encoder']}")
    print(f"  ✓ Parameters: {total_params:,} ({trainable_params:,} trainable)")
    print(f"  ✓ Model size: ~{total_params * 4 / 1e6:.2f} MB")
    print(f"  ✓ Best epoch: {checkpoint['epoch'] + 1}")
    print(f"  ✓ Val Dice at save: {checkpoint['val_dice']:.4f}\n")

    return model, checkpoint

# Load all three models
print(f"\n{'='*70}")
print("LOADING ALL TRAINED MODELS".center(70))
print(f"{'='*70}\n")

models = {}
checkpoints = {}

for model_id, config in MODEL_CONFIGS.items():
    models[model_id], checkpoints[model_id] = load_model(config)

print(f"✅ All models loaded successfully!\n")

In [ ]:
# ============================================================================
# PHASE 1: COMPREHENSIVE TEST SET EVALUATION
# ============================================================================

print(f"\n{'='*70}")
print("PHASE 1: TEST SET EVALUATION".center(70))
print(f"{'='*70}\n")

def calculate_metrics(predictions, targets, num_classes=3):
    """Calculate comprehensive metrics"""
    predictions = predictions.long()
    targets = targets.long()

    metrics = {}

    for c in range(num_classes):
        pred_c = (predictions == c).float()
        target_c = (targets == c).float()

        # True Positives, False Positives, True Negatives, False Negatives
        tp = (pred_c * target_c).sum().item()
        fp = (pred_c * (1 - target_c)).sum().item()
        tn = ((1 - pred_c) * (1 - target_c)).sum().item()
        fn = ((1 - pred_c) * target_c).sum().item()

        # Dice Score
        dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)

        # IoU (Jaccard Index)
        iou = tp / (tp + fp + fn + 1e-8)

        # Sensitivity (Recall / True Positive Rate)
        sensitivity = tp / (tp + fn + 1e-8)

        # Specificity (True Negative Rate)
        specificity = tn / (tn + fp + 1e-8)

        # Precision
        precision = tp / (tp + fp + 1e-8)

        # F1 Score (same as Dice for binary)
        f1 = 2 * precision * sensitivity / (precision + sensitivity + 1e-8)

        metrics[f'class_{c}'] = {
            'dice': dice,
            'iou': iou,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'precision': precision,
            'f1': f1
        }

    # Overall metrics (arteries + veins only, excluding background)
    metrics['overall'] = {
        'dice': (metrics['class_1']['dice'] + metrics['class_2']['dice']) / 2,
        'iou': (metrics['class_1']['iou'] + metrics['class_2']['iou']) / 2,
        'sensitivity': (metrics['class_1']['sensitivity'] + metrics['class_2']['sensitivity']) / 2,
        'specificity': (metrics['class_1']['specificity'] + metrics['class_2']['specificity']) / 2,
        'precision': (metrics['class_1']['precision'] + metrics['class_2']['precision']) / 2,
        'f1': (metrics['class_1']['f1'] + metrics['class_2']['f1']) / 2,
    }

    return metrics

def evaluate_model_on_test(model, test_loader, device, model_name):
    """Evaluate model comprehensively on test set"""
    model.eval()

    all_metrics = []
    all_predictions = []
    all_targets = []
    all_images = []
    all_ids = []
    all_datasets = []

    print(f"Evaluating {model_name}...")

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"  Testing"):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            # Forward pass
            outputs = model(images)
            preds = torch.softmax(outputs, dim=1)
            preds = torch.argmax(preds, dim=1)

            # Calculate metrics for this batch
            batch_metrics = calculate_metrics(preds, masks)
            all_metrics.append(batch_metrics)

            # Store for visualization
            all_predictions.append(preds.cpu().numpy())
            all_targets.append(masks.cpu().numpy())
            all_images.append(images.cpu().numpy())
            all_ids.extend(batch['id'])
            all_datasets.extend(batch['dataset'])

    # Aggregate metrics
    aggregated_metrics = defaultdict(lambda: defaultdict(list))

    for batch_metric in all_metrics:
        for key, values in batch_metric.items():
            for metric_name, metric_value in values.items():
                aggregated_metrics[key][metric_name].append(metric_value)

    # Calculate mean and std
    final_metrics = {}
    for key, metrics_dict in aggregated_metrics.items():
        final_metrics[key] = {}
        for metric_name, values in metrics_dict.items():
            final_metrics[key][metric_name] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values)
            }

    return final_metrics, all_predictions, all_targets, all_images, all_ids, all_datasets

# Evaluate all models
all_results = {}
all_predictions_dict = {}
all_targets_dict = {}
all_images_dict = {}
all_ids_dict = {}
all_datasets_dict = {}

for model_id, config in MODEL_CONFIGS.items():
    results, preds, targets, images, ids, datasets = evaluate_model_on_test(
        models[model_id],
        test_loader,
        device,
        config['name']
    )

    all_results[model_id] = results
    all_predictions_dict[model_id] = preds
    all_targets_dict[model_id] = targets
    all_images_dict[model_id] = images
    all_ids_dict[model_id] = ids
    all_datasets_dict[model_id] = datasets

print(f"\n✅ Phase 1 Complete: All models evaluated\n")

In [ ]:
# ============================================================================
# CREATE RESULTS COMPARISON TABLE
# ============================================================================

print(f"\n{'='*80}")
print("MODEL PERFORMANCE COMPARISON".center(80))
print(f"{'='*80}\n")

def create_comparison_table(all_results, model_configs):
    rows = []

    for model_id, config in model_configs.items():
        results = all_results[model_id]

        row = {
            'Model': config['name'],
            'Architecture': config['architecture'],
            'Encoder': config['encoder'],

            # Overall metrics (mean only, cleaner)
            'Dice': results['overall']['dice']['mean'],
            'Dice_std': results['overall']['dice']['std'],
            'IoU': results['overall']['iou']['mean'],
            'IoU_std': results['overall']['iou']['std'],
            'Sensitivity': results['overall']['sensitivity']['mean'],
            'Sens_std': results['overall']['sensitivity']['std'],
            'Specificity': results['overall']['specificity']['mean'],
            'Spec_std': results['overall']['specificity']['std'],
            'Precision': results['overall']['precision']['mean'],
            'Prec_std': results['overall']['precision']['std'],

            # Class-specific
            'BG_Dice': results['class_0']['dice']['mean'],
            'Artery_Dice': results['class_1']['dice']['mean'],
            'Artery_Dice_std': results['class_1']['dice']['std'],
            'Artery_IoU': results['class_1']['iou']['mean'],
            'Artery_Sens': results['class_1']['sensitivity']['mean'],
            'Artery_Prec': results['class_1']['precision']['mean'],
            'Vein_Dice': results['class_2']['dice']['mean'],
            'Vein_Dice_std': results['class_2']['dice']['std'],
            'Vein_IoU': results['class_2']['iou']['mean'],
            'Vein_Sens': results['class_2']['sensitivity']['mean'],
            'Vein_Prec': results['class_2']['precision']['mean'],
        }

        rows.append(row)

    return pd.DataFrame(rows)


def print_formatted_table(df, title, columns_config):

    print(f"\n{'─'*80}")
    print(f"  {title}")
    print(f"{'─'*80}")

    # Print header
    header = ""
    for col, width in columns_config:
        if col in df.columns:
            header += f"{col:<{width}}"
    print(header)
    print(f"{'─'*80}")

    # Print rows
    for idx, row in df.iterrows():
        line = ""
        for col, width in columns_config:
            if col in df.columns:
                value = row[col]
                if isinstance(value, float):
                    line += f"{value:<{width}.4f}"
                else:
                    line += f"{str(value):<{width}}"
        print(line)

    print(f"{'─'*80}\n")


# Create main dataframe
comparison_df = create_comparison_table(all_results, MODEL_CONFIGS)

# ============================================================================
# TABLE 1: OVERALL PERFORMANCE SUMMARY
# ============================================================================

summary_data = []
for idx, row in comparison_df.iterrows():
    summary_data.append({
        'Model': row['Model'],
        'Dice Score': f"{row['Dice']:.4f} ± {row['Dice_std']:.4f}",
        'IoU': f"{row['IoU']:.4f} ± {row['IoU_std']:.4f}",
        'Sensitivity': f"{row['Sensitivity']:.4f} ± {row['Sens_std']:.4f}",
        'Specificity': f"{row['Specificity']:.4f} ± {row['Spec_std']:.4f}",
        'Precision': f"{row['Precision']:.4f} ± {row['Prec_std']:.4f}"
    })

summary_df = pd.DataFrame(summary_data)

print(f"\n{'═'*80}")
print("TABLE 1: OVERALL PERFORMANCE METRICS".center(80))
print(f"{'═'*80}")
print(summary_df.to_string(index=False))
print(f"{'═'*80}\n")

# ============================================================================
# TABLE 2: CLASS-SPECIFIC PERFORMANCE (ARTERIES)
# ============================================================================

artery_data = []
for idx, row in comparison_df.iterrows():
    artery_data.append({
        'Model': row['Model'],
        'Artery Dice': f"{row['Artery_Dice']:.4f} ± {row['Artery_Dice_std']:.4f}",
        'Artery IoU': f"{row['Artery_IoU']:.4f}",
        'Artery Sensitivity': f"{row['Artery_Sens']:.4f}",
        'Artery Precision': f"{row['Artery_Prec']:.4f}"
    })

artery_df = pd.DataFrame(artery_data)

print(f"{'═'*80}")
print("TABLE 2: ARTERY SEGMENTATION PERFORMANCE".center(80))
print(f"{'═'*80}")
print(artery_df.to_string(index=False))
print(f"{'═'*80}\n")

# ============================================================================
# TABLE 3: CLASS-SPECIFIC PERFORMANCE (VEINS)
# ============================================================================

vein_data = []
for idx, row in comparison_df.iterrows():
    vein_data.append({
        'Model': row['Model'],
        'Vein Dice': f"{row['Vein_Dice']:.4f} ± {row['Vein_Dice_std']:.4f}",
        'Vein IoU': f"{row['Vein_IoU']:.4f}",
        'Vein Sensitivity': f"{row['Vein_Sens']:.4f}",
        'Vein Precision': f"{row['Vein_Prec']:.4f}"
    })

vein_df = pd.DataFrame(vein_data)

print(f"{'═'*80}")
print("TABLE 3: VEIN SEGMENTATION PERFORMANCE".center(80))
print(f"{'═'*80}")
print(vein_df.to_string(index=False))
print(f"{'═'*80}\n")

# ============================================================================
# TABLE 4: COMPACT COMPARISON (FOR REPORTS/PAPERS)
# ============================================================================

compact_data = []
for idx, row in comparison_df.iterrows():
    compact_data.append({
        'Model': row['Model'],
        'Overall Dice': f"{row['Dice']:.4f}",
        'Artery Dice': f"{row['Artery_Dice']:.4f}",
        'Vein Dice': f"{row['Vein_Dice']:.4f}",
        'Background Dice': f"{row['BG_Dice']:.4f}",
        'Mean IoU': f"{row['IoU']:.4f}"
    })

compact_df = pd.DataFrame(compact_data)

print(f"{'═'*80}")
print("TABLE 4: COMPACT COMPARISON (Quick Reference)".center(80))
print(f"{'═'*80}")
print(compact_df.to_string(index=False))
print(f"{'═'*80}\n")

# ============================================================================
# TABLE 5: RANKING TABLE
# ============================================================================

ranking_data = []
metrics_to_rank = [
    ('Dice', 'Overall Dice'),
    ('Artery_Dice', 'Artery Dice'),
    ('Vein_Dice', 'Vein Dice'),
    ('IoU', 'Overall IoU'),
    ('Sensitivity', 'Sensitivity'),
    ('Specificity', 'Specificity')
]

for metric_key, metric_name in metrics_to_rank:
    sorted_models = comparison_df.sort_values(metric_key, ascending=False)
    ranking_data.append({
        'Metric': metric_name,
        '1st(Best model)': f"{sorted_models.iloc[0]['Model']} ({sorted_models.iloc[0][metric_key]:.4f})",
        '2nd': f"{sorted_models.iloc[1]['Model']} ({sorted_models.iloc[1][metric_key]:.4f})",
        '3rd': f"{sorted_models.iloc[2]['Model']} ({sorted_models.iloc[2][metric_key]:.4f})"
    })

ranking_df = pd.DataFrame(ranking_data)

print(f"{'═'*80}")
print("TABLE 5: MODEL RANKINGS BY METRIC".center(80))
print(f"{'═'*80}")
print(ranking_df.to_string(index=False))
print(f"{'═'*80}\n")

# ============================================================================
# SAVE ALL TABLES
# ============================================================================

print(f"{'='*80}")
print("SAVING TABLES".center(80))
print(f"{'='*80}\n")

tables_to_save = {
    'overall_performance': summary_df,
    'artery_performance': artery_df,
    'vein_performance': vein_df,
    'compact_comparison': compact_df,
    'rankings': ranking_df,
    'full_results': comparison_df
}

for name, df in tables_to_save.items():
    filepath = f"{OUTPUT_DIR}/tables/{name}.csv"
    df.to_csv(filepath, index=False)
    print(f"  ✓ Saved: {name}.csv")

print(f"\n{'='*80}")
print("✅ ALL TABLES SAVED SUCCESSFULLY".center(80))
print(f"{'='*80}\n")

# ============================================================================
# PERFORMANCE HIGHLIGHTS
# ============================================================================

print(f"{'═'*80}")
print("PERFORMANCE HIGHLIGHTS".center(80))
print(f"{'═'*80}\n")

best_overall = comparison_df.loc[comparison_df['Dice'].idxmax()]
best_artery = comparison_df.loc[comparison_df['Artery_Dice'].idxmax()]
best_vein = comparison_df.loc[comparison_df['Vein_Dice'].idxmax()]
best_specificity = comparison_df.loc[comparison_df['Specificity'].idxmax()]

print(f"   Best Overall Performance:  {best_overall['Model']}")
print(f"     └─ Dice Score: {best_overall['Dice']:.4f} ± {best_overall['Dice_std']:.4f}\n")

print(f"   Best Artery Segmentation:  {best_artery['Model']}")
print(f"     └─ Artery Dice: {best_artery['Artery_Dice']:.4f} ± {best_artery['Artery_Dice_std']:.4f}\n")

print(f"   Best Vein Segmentation:    {best_vein['Model']}")
print(f"     └─ Vein Dice: {best_vein['Vein_Dice']:.4f} ± {best_vein['Vein_Dice_std']:.4f}\n")

print(f"   Best Specificity:          {best_specificity['Model']}")
print(f"     └─ Specificity: {best_specificity['Specificity']:.4f} ± {best_specificity['Spec_std']:.4f}\n")

print(f"{'═'*80}\n")

# ============================================================================
# IMPROVEMENT ANALYSIS
# ============================================================================

print(f"{'═'*80}")
print("RELATIVE IMPROVEMENTS".center(80))
print(f"{'═'*80}\n")

baseline_model = comparison_df.iloc[0]  # First model as baseline
print(f"Baseline Model: {baseline_model['Model']}\n")

for idx, row in comparison_df.iloc[1:].iterrows():
    dice_improvement = ((row['Dice'] - baseline_model['Dice']) / baseline_model['Dice']) * 100
    artery_improvement = ((row['Artery_Dice'] - baseline_model['Artery_Dice']) / baseline_model['Artery_Dice']) * 100
    vein_improvement = ((row['Vein_Dice'] - baseline_model['Vein_Dice']) / baseline_model['Vein_Dice']) * 100

    print(f"  {row['Model']}:")
    print(f"    • Overall Dice:  {dice_improvement:+.2f}% ({row['Dice']:.4f} vs {baseline_model['Dice']:.4f})")
    print(f"    • Artery Dice:   {artery_improvement:+.2f}% ({row['Artery_Dice']:.4f} vs {baseline_model['Artery_Dice']:.4f})")
    print(f"    • Vein Dice:     {vein_improvement:+.2f}% ({row['Vein_Dice']:.4f} vs {baseline_model['Vein_Dice']:.4f})")
    print()

print(f"{'═'*80}\n")

In [ ]:
# ============================================================================
# VISUALIZE METRIC COMPARISONS
# ============================================================================

print(f"Creating comparison visualizations...\n")

def plot_metric_comparison(all_results, model_configs, save_dir):
    """Create bar plots comparing key metrics"""

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Model Performance Comparison on Test Set', fontsize=16, fontweight='bold')

    metrics_to_plot = [
        ('overall', 'dice', 'Overall Dice Score'),
        ('overall', 'iou', 'Overall IoU'),
        ('overall', 'sensitivity', 'Overall Sensitivity'),
        ('class_1', 'dice', 'Artery Dice Score'),
        ('class_2', 'dice', 'Vein Dice Score'),
        ('overall', 'precision', 'Overall Precision')
    ]

    for idx, (class_key, metric_key, title) in enumerate(metrics_to_plot):
        ax = axes[idx // 3, idx % 3]

        model_names = []
        means = []
        stds = []
        colors = []

        for model_id, config in model_configs.items():
            results = all_results[model_id]
            model_names.append(config['name'])
            means.append(results[class_key][metric_key]['mean'])
            stds.append(results[class_key][metric_key]['std'])
            colors.append(config['color'])

        bars = ax.bar(range(len(model_names)), means, yerr=stds,
                      color=colors, alpha=0.8, capsize=5, edgecolor='black', linewidth=1.5)

        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.set_ylabel('Score', fontsize=10)
        ax.set_xticks(range(len(model_names)))
        ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
        ax.set_ylim([0, 1.0])
        ax.grid(axis='y', alpha=0.3, linestyle='--')

        # Add value labels on bars
        for i, (bar, mean, std) in enumerate(zip(bars, means, stds)):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + std + 0.02,
                   f'{mean:.3f}',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/metric_comparison_bars.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"  ✓ Saved: metric_comparison_bars.png")

plot_metric_comparison(all_results, MODEL_CONFIGS, f"{OUTPUT_DIR}/figures")

In [ ]:
# ============================================================================
# PHASE 2: QUALITATIVE VISUAL ANALYSIS
# ============================================================================

print(f"\n{'='*70}")
print("PHASE 2: VISUAL ANALYSIS".center(70))
print(f"{'='*70}\n")

def denormalize_image(img):
    """Convert normalized tensor back to displayable image"""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

def mask_to_color(mask):
    """Convert class mask to RGB color image"""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    # Background: black
    color_mask[mask == 0] = [0, 0, 0]
    # Artery: red
    color_mask[mask == 1] = [255, 0, 0]
    # Vein: blue
    color_mask[mask == 2] = [0, 0, 255]

    return color_mask

def create_overlay(image, mask, alpha=0.5):
    """Create overlay of mask on image"""
    color_mask = mask_to_color(mask)
    # Convert image to 0-255 range
    image_uint8 = (image * 255).astype(np.uint8)
    overlay = cv2.addWeighted(image_uint8, 1-alpha, color_mask, alpha, 0)
    return overlay

def visualize_predictions(sample_idx, all_images_dict, all_targets_dict,
                         all_predictions_dict, all_ids_dict, model_configs, save_dir):
    """Visualize predictions from all models for a single sample"""

    # Get the sample (use model1 as reference since all have same test set)
    batch_idx = sample_idx // BATCH_SIZE
    within_batch_idx = sample_idx % BATCH_SIZE

    image = all_images_dict['model1'][batch_idx][within_batch_idx]
    target = all_targets_dict['model1'][batch_idx][within_batch_idx]
    sample_id = all_ids_dict['model1'][sample_idx]

    # Denormalize image
    image_display = denormalize_image(image)

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Sample: {sample_id}', fontsize=16, fontweight='bold')

    # Original image
    axes[0, 0].imshow(image_display)
    axes[0, 0].set_title('Original Image', fontweight='bold')
    axes[0, 0].axis('off')

    # Ground truth
    gt_color = mask_to_color(target)
    axes[0, 1].imshow(gt_color)
    axes[0, 1].set_title('Ground Truth', fontweight='bold')
    axes[0, 1].axis('off')

    # Ground truth overlay
    gt_overlay = create_overlay(image_display, target, alpha=0.4)
    axes[0, 2].imshow(gt_overlay)
    axes[0, 2].set_title('GT Overlay', fontweight='bold')
    axes[0, 2].axis('off')

    # Predictions from each model
    for idx, (model_id, config) in enumerate(MODEL_CONFIGS.items()):
        pred = all_predictions_dict[model_id][batch_idx][within_batch_idx]
        pred_overlay = create_overlay(image_display, pred, alpha=0.4)

        # Calculate Dice for this sample
        pred_tensor = torch.from_numpy(pred)
        target_tensor = torch.from_numpy(target)
        metrics = calculate_metrics(pred_tensor, target_tensor)
        dice_score = metrics['overall']['dice']

        axes[1, idx].imshow(pred_overlay)
        axes[1, idx].set_title(f"{config['name']}\nDice: {dice_score:.4f}",
                              fontweight='bold', fontsize=10)
        axes[1, idx].axis('off')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/sample_{sample_idx:03d}_{sample_id}.png",
                dpi=200, bbox_inches='tight')
    plt.close()

# Select diverse samples to visualize
print("Generating visual predictions for sample images...")

# Calculate per-sample dice scores for model1 to find best/worst/median
sample_dice_scores = []
for batch_idx in range(len(all_predictions_dict['model1'])):
    for within_idx in range(len(all_predictions_dict['model1'][batch_idx])):
        pred = all_predictions_dict['model1'][batch_idx][within_idx]
        target = all_targets_dict['model1'][batch_idx][within_idx]

        pred_tensor = torch.from_numpy(pred)
        target_tensor = torch.from_numpy(target)
        metrics = calculate_metrics(pred_tensor, target_tensor)

        sample_idx = batch_idx * BATCH_SIZE + within_idx
        sample_dice_scores.append((sample_idx, metrics['overall']['dice']))

# Sort by dice score
sample_dice_scores.sort(key=lambda x: x[1])

# Select samples: worst 2, median 2, best 2
num_samples = len(sample_dice_scores)
selected_samples = [
    sample_dice_scores[0][0],  # Worst
    sample_dice_scores[1][0],  # 2nd worst
    sample_dice_scores[num_samples//2 - 1][0],  # Median 1
    sample_dice_scores[num_samples//2][0],  # Median 2
    sample_dice_scores[-2][0],  # 2nd best
    sample_dice_scores[-1][0],  # Best
]

for idx, sample_idx in enumerate(selected_samples):
    print(f"  Generating visualization {idx+1}/6...")
    visualize_predictions(
        sample_idx,
        all_images_dict,
        all_targets_dict,
        all_predictions_dict,
        all_ids_dict,
        MODEL_CONFIGS,
        f"{OUTPUT_DIR}/predictions"
    )

print(f"\n✅ Phase 2 Complete: Visual analysis generated")
print(f"   Saved 6 prediction visualizations to: {OUTPUT_DIR}/predictions/\n")

In [ ]:
# ============================================================================
# ERROR ANALYSIS
# ============================================================================

print(f"Performing error analysis...\n")

def analyze_errors(all_predictions_dict, all_targets_dict, model_configs):
    """Analyze where models make errors"""

    error_analysis = {}

    for model_id, config in model_configs.items():
        predictions = all_predictions_dict[model_id]
        targets = all_targets_dict[model_id]

        total_pixels = 0
        correct_pixels = 0
        class_errors = {0: 0, 1: 0, 2: 0}  # Errors per class
        confusion_matrix = np.zeros((3, 3), dtype=int)  # 3x3 for 3 classes

        for batch_preds, batch_targets in zip(predictions, targets):
            for pred, target in zip(batch_preds, batch_targets):
                total_pixels += pred.size
                correct_pixels += (pred == target).sum()

                # Count errors per class
                for c in range(3):
                    class_mask = (target == c)
                    errors = (pred[class_mask] != c).sum()
                    class_errors[c] += errors

                # Build confusion matrix
                for true_class in range(3):
                    for pred_class in range(3):
                        confusion_matrix[true_class, pred_class] += \
                            ((target == true_class) & (pred == pred_class)).sum()

        pixel_accuracy = correct_pixels / total_pixels

        error_analysis[model_id] = {
            'pixel_accuracy': pixel_accuracy,
            'class_errors': class_errors,
            'confusion_matrix': confusion_matrix
        }

    return error_analysis

error_results = analyze_errors(all_predictions_dict, all_targets_dict, MODEL_CONFIGS)

# Print error analysis
print(f"{'='*70}")
print("ERROR ANALYSIS".center(70))
print(f"{'='*70}\n")

for model_id, config in MODEL_CONFIGS.items():
    print(f"\n{config['name']}:")
    print(f"  Pixel Accuracy: {error_results[model_id]['pixel_accuracy']:.4f}")
    print(f"  Errors by class:")
    print(f"    Background errors: {error_results[model_id]['class_errors'][0]:,}")
    print(f"    Artery errors: {error_results[model_id]['class_errors'][1]:,}")
    print(f"    Vein errors: {error_results[model_id]['class_errors'][2]:,}")

print()

# Visualize confusion matrices
def plot_confusion_matrices(error_results, model_configs, save_dir):
    """Plot confusion matrices for all models"""

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Confusion Matrices - Test Set', fontsize=16, fontweight='bold')

    class_names = ['Background', 'Artery', 'Vein']

    for idx, (model_id, config) in enumerate(model_configs.items()):
        cm = error_results[model_id]['confusion_matrix']

        # Normalize confusion matrix
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

        # Plot
        im = axes[idx].imshow(cm_normalized, cmap='cividis', aspect='auto', vmin=0, vmax=1)
        axes[idx].set_title(config['name'], fontweight='bold', fontsize=12)
        axes[idx].set_xlabel('Predicted', fontsize=10)
        axes[idx].set_ylabel('True', fontsize=10)
        axes[idx].set_xticks(range(3))
        axes[idx].set_yticks(range(3))
        axes[idx].set_xticklabels(class_names, rotation=45, ha='right')
        axes[idx].set_yticklabels(class_names)

        # Add text annotations
        for i in range(3):
            for j in range(3):
                text_color = 'white' if cm_normalized[i, j] > 0.5 else 'black'
                axes[idx].text(j, i, f'{cm_normalized[i, j]:.3f}\n({cm[i, j]:,})',
                             ha='center', va='center', color=text_color, fontsize=9)

        # Add colorbar
        if idx == 2:
            cbar = plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
            cbar.set_label('Normalized Frequency', rotation=270, labelpad=20)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/confusion_matrices.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"  ✓ Saved: confusion_matrices.png")

plot_confusion_matrices(error_results, MODEL_CONFIGS, f"{OUTPUT_DIR}/figures")

print(f"\n✅ Error analysis complete\n")

In [ ]:
# ============================================================================
# TRAINING HISTORY COMPARISON
# ============================================================================

print(f"{'='*70}")
print("TRAINING HISTORY VISUALIZATION".center(70))
print(f"{'='*70}\n")

# Load training histories
try:
    history_model1 = pd.read_csv('/kaggle/input/unet-resnet34/model1_outputs/model1_history.csv')
    history_model2 = pd.read_csv('/kaggle/input/unet-resnet50/model2_outputs/model2_history.csv')
    history_model3 = pd.read_csv('/kaggle/input/unet-efficientnet-b4/model3_outputs/model3_history.csv')

    histories_loaded = True
    print("✓ Training histories loaded from CSV files\n")
except:
    print("⚠️  Training history CSV files not found. Skipping history visualization.")
    print("   (Make sure to save training history during training)\n")
    histories_loaded = False

if histories_loaded:
    def plot_training_history(hist1, hist2, hist3, model_configs, save_dir):
        """Plot comprehensive training history comparison"""

        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

        fig.suptitle('Training History Comparison - All Models', fontsize=18, fontweight='bold')

        # Get model info for legend
        models_info = [
            (hist1, MODEL_CONFIGS['model1']),
            (hist2, MODEL_CONFIGS['model2']),
            (hist3, MODEL_CONFIGS['model3'])
        ]

        # 1. Training Loss
        ax1 = fig.add_subplot(gs[0, 0])
        for hist, config in models_info:
            ax1.plot(hist['train_loss'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax1.set_title('Training Loss', fontweight='bold', fontsize=12)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend(loc='best', fontsize=8)
        ax1.grid(True, alpha=0.3)

        # 2. Validation Loss
        ax2 = fig.add_subplot(gs[0, 1])
        for hist, config in models_info:
            ax2.plot(hist['val_loss'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax2.set_title('Validation Loss', fontweight='bold', fontsize=12)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend(loc='best', fontsize=8)
        ax2.grid(True, alpha=0.3)

        # 3. Learning Rate
        ax3 = fig.add_subplot(gs[0, 2])
        for hist, config in models_info:
            ax3.plot(hist['lr'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax3.set_title('Learning Rate Schedule', fontweight='bold', fontsize=12)
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Learning Rate')
        ax3.set_yscale('log')
        ax3.legend(loc='best', fontsize=8)
        ax3.grid(True, alpha=0.3)

        # 4. Training Dice
        ax4 = fig.add_subplot(gs[1, 0])
        for hist, config in models_info:
            ax4.plot(hist['train_dice'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax4.set_title('Training Dice Score', fontweight='bold', fontsize=12)
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('Dice Score')
        ax4.legend(loc='best', fontsize=8)
        ax4.grid(True, alpha=0.3)

        # 5. Validation Dice (Overall)
        ax5 = fig.add_subplot(gs[1, 1])
        for hist, config in models_info:
            ax5.plot(hist['val_dice'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax5.set_title('Validation Dice Score (Overall)', fontweight='bold', fontsize=12)
        ax5.set_xlabel('Epoch')
        ax5.set_ylabel('Dice Score')
        ax5.legend(loc='best', fontsize=8)
        ax5.grid(True, alpha=0.3)

        # 6. Background Dice
        ax6 = fig.add_subplot(gs[1, 2])
        for hist, config in models_info:
            ax6.plot(hist['val_dice_bg'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax6.set_title('Background Dice Score', fontweight='bold', fontsize=12)
        ax6.set_xlabel('Epoch')
        ax6.set_ylabel('Dice Score')
        ax6.legend(loc='best', fontsize=8)
        ax6.grid(True, alpha=0.3)

        # 7. Artery Dice
        ax7 = fig.add_subplot(gs[2, 0])
        for hist, config in models_info:
            ax7.plot(hist['val_dice_artery'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax7.set_title('Artery Dice Score', fontweight='bold', fontsize=12)
        ax7.set_xlabel('Epoch')
        ax7.set_ylabel('Dice Score')
        ax7.legend(loc='best', fontsize=8)
        ax7.grid(True, alpha=0.3)

        # 8. Vein Dice
        ax8 = fig.add_subplot(gs[2, 1])
        for hist, config in models_info:
            ax8.plot(hist['val_dice_vein'], label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax8.set_title('Vein Dice Score', fontweight='bold', fontsize=12)
        ax8.set_xlabel('Epoch')
        ax8.set_ylabel('Dice Score')
        ax8.legend(loc='best', fontsize=8)
        ax8.grid(True, alpha=0.3)

        # 9. Loss Difference (Train - Val)
        ax9 = fig.add_subplot(gs[2, 2])
        for hist, config in models_info:
            loss_diff = hist['train_loss'] - hist['val_loss']
            ax9.plot(loss_diff, label=config['name'],
                    color=config['color'], linewidth=2, marker=config['marker'],
                    markevery=len(hist)//10, markersize=5)
        ax9.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax9.set_title('Overfitting Monitor (Train Loss - Val Loss)', fontweight='bold', fontsize=12)
        ax9.set_xlabel('Epoch')
        ax9.set_ylabel('Loss Difference')
        ax9.legend(loc='best', fontsize=8)
        ax9.grid(True, alpha=0.3)

        plt.savefig(f"{save_dir}/training_history_comprehensive.png", dpi=300, bbox_inches='tight')
        plt.show()

        print(f"  ✓ Saved: training_history_comprehensive.png")

    plot_training_history(history_model1, history_model2, history_model3,
                         MODEL_CONFIGS, f"{OUTPUT_DIR}/figures")

    print(f"\n✅ Training history visualization complete\n")

In [ ]:
# ============================================================================
# PHASE 3: INFERENCE SPEED & EFFICIENCY ANALYSIS
# ============================================================================

print(f"\n{'='*70}")
print("PHASE 3: EFFICIENCY & SPEED ANALYSIS".center(70))
print(f"{'='*70}\n")

def measure_inference_speed(model, test_loader, device, num_iterations=100):
    """Measure inference speed and throughput"""
    model.eval()

    # Warmup
    print("  Warming up GPU...")
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if i >= 5:
                break
            images = batch['image'].to(device)
            _ = model(images)

    # Actual timing
    print("  Measuring inference time...")
    times = []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(test_loader, desc="  Timing", total=min(num_iterations, len(test_loader)))):
            if i >= num_iterations:
                break

            images = batch['image'].to(device)

            # Time inference
            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start_time = time.time()
            outputs = model(images)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end_time = time.time()

            batch_time = end_time - start_time
            per_image_time = batch_time / images.shape[0]
            times.append(per_image_time)

    mean_time = np.mean(times)
    std_time = np.std(times)
    fps = 1.0 / mean_time

    return {
        'mean_time_ms': mean_time * 1000,
        'std_time_ms': std_time * 1000,
        'fps': fps,
        'throughput_imgs_per_sec': fps
    }

def get_model_size(model):
    """Calculate model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    size_mb = (param_size + buffer_size) / (1024 ** 2)
    return size_mb

def get_model_parameters(model):
    """Get parameter counts"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

# Measure efficiency for all models
efficiency_results = {}

for model_id, config in MODEL_CONFIGS.items():
    print(f"\nAnalyzing {config['name']}...")

    # Speed
    speed_metrics = measure_inference_speed(models[model_id], test_loader, device, num_iterations=50)

    # Size
    model_size_mb = get_model_size(models[model_id])

    # Parameters
    total_params, trainable_params = get_model_parameters(models[model_id])

    # GPU Memory (if available)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            batch = next(iter(test_loader))
            images = batch['image'].to(device)
            _ = models[model_id](images)
        gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        gpu_memory_mb = None

    efficiency_results[model_id] = {
        'inference_time_ms': speed_metrics['mean_time_ms'],
        'inference_std_ms': speed_metrics['std_time_ms'],
        'fps': speed_metrics['fps'],
        'model_size_mb': model_size_mb,
        'total_params': total_params,
        'trainable_params': trainable_params,
        'gpu_memory_mb': gpu_memory_mb
    }

    print(f"  ✓ Inference time: {speed_metrics['mean_time_ms']:.2f} ± {speed_metrics['std_time_ms']:.2f} ms")
    print(f"  ✓ FPS: {speed_metrics['fps']:.2f}")
    print(f"  ✓ Model size: {model_size_mb:.2f} MB")
    print(f"  ✓ Parameters: {total_params:,}")
    if gpu_memory_mb:
        print(f"  ✓ GPU Memory: {gpu_memory_mb:.2f} MB")

# Create efficiency comparison table
print(f"\n{'='*70}")
print("EFFICIENCY COMPARISON TABLE".center(70))
print(f"{'='*70}\n")

efficiency_rows = []
for model_id, config in MODEL_CONFIGS.items():
    row = {
        'Model': config['name'],
        'Inference Time (ms)': f"{efficiency_results[model_id]['inference_time_ms']:.2f} ± {efficiency_results[model_id]['inference_std_ms']:.2f}",
        'FPS': f"{efficiency_results[model_id]['fps']:.2f}",
        'Model Size (MB)': f"{efficiency_results[model_id]['model_size_mb']:.2f}",
        'Parameters (M)': f"{efficiency_results[model_id]['total_params'] / 1e6:.2f}",
    }

    if efficiency_results[model_id]['gpu_memory_mb']:
        row['GPU Memory (MB)'] = f"{efficiency_results[model_id]['gpu_memory_mb']:.2f}"

    efficiency_rows.append(row)

efficiency_df = pd.DataFrame(efficiency_rows)
print(efficiency_df.to_string(index=False))
print(f"{'='*70}\n")

efficiency_df.to_csv(f"{OUTPUT_DIR}/tables/efficiency_comparison.csv", index=False)
print(f"✅ Efficiency table saved to: {OUTPUT_DIR}/tables/efficiency_comparison.csv\n")

# Visualize efficiency trade-offs
def plot_efficiency_tradeoffs(all_results, efficiency_results, model_configs, save_dir):
    """Plot accuracy vs efficiency trade-offs"""

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Performance vs Efficiency Trade-offs', fontsize=16, fontweight='bold')

    models_data = []
    for model_id, config in model_configs.items():
        models_data.append({
            'name': config['name'],
            'dice': all_results[model_id]['overall']['dice']['mean'],
            'time': efficiency_results[model_id]['inference_time_ms'],
            'params': efficiency_results[model_id]['total_params'] / 1e6,
            'size': efficiency_results[model_id]['model_size_mb'],
            'color': config['color']
        })

    # 1. Dice vs Inference Time
    ax1 = axes[0]
    for data in models_data:
        ax1.scatter(data['time'], data['dice'], s=200, c=data['color'],
                   edgecolors='black', linewidth=2, alpha=0.7, zorder=3)
        ax1.annotate(data['name'], (data['time'], data['dice']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax1.set_xlabel('Inference Time (ms)', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Dice Score', fontsize=11, fontweight='bold')
    ax1.set_title('Accuracy vs Speed', fontweight='bold')
    ax1.grid(True, alpha=0.3)

    # 2. Dice vs Parameters
    ax2 = axes[1]
    for data in models_data:
        ax2.scatter(data['params'], data['dice'], s=200, c=data['color'],
                   edgecolors='black', linewidth=2, alpha=0.7, zorder=3)
        ax2.annotate(data['name'], (data['params'], data['dice']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax2.set_xlabel('Parameters (Millions)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Dice Score', fontsize=11, fontweight='bold')
    ax2.set_title('Accuracy vs Model Size', fontweight='bold')
    ax2.grid(True, alpha=0.3)

    # 3. Parameters vs Inference Time
    ax3 = axes[2]
    for data in models_data:
        ax3.scatter(data['params'], data['time'], s=200, c=data['color'],
                   edgecolors='black', linewidth=2, alpha=0.7, zorder=3)
        ax3.annotate(data['name'], (data['params'], data['time']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax3.set_xlabel('Parameters (Millions)', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Inference Time (ms)', fontsize=11, fontweight='bold')
    ax3.set_title('Model Size vs Speed', fontweight='bold')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/efficiency_tradeoffs.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"  ✓ Saved: efficiency_tradeoffs.png")

plot_efficiency_tradeoffs(all_results, efficiency_results, MODEL_CONFIGS, f"{OUTPUT_DIR}/figures")

print(f"\n✅ Phase 3 Complete: Efficiency analysis done\n")

In [ ]:
# ============================================================================
# PHASE 4: STATISTICAL VALIDATION
# ============================================================================

print(f"\n{'='*70}")
print("PHASE 4: STATISTICAL VALIDATION".center(70))
print(f"{'='*70}\n")

# Per-dataset performance breakdown
def analyze_per_dataset_performance(all_predictions_dict, all_targets_dict,
                                   all_datasets_dict, model_configs):
    """Analyze performance breakdown by dataset"""

    dataset_performance = {}

    for model_id, config in model_configs.items():
        predictions = all_predictions_dict[model_id]
        targets = all_targets_dict[model_id]
        datasets = all_datasets_dict[model_id]

        # Group by dataset
        dataset_metrics = defaultdict(list)

        sample_idx = 0
        for batch_preds, batch_targets in zip(predictions, targets):
            for pred, target in zip(batch_preds, batch_targets):
                dataset_name = datasets[sample_idx]

                pred_tensor = torch.from_numpy(pred)
                target_tensor = torch.from_numpy(target)
                metrics = calculate_metrics(pred_tensor, target_tensor)

                dataset_metrics[dataset_name].append(metrics['overall']['dice'])
                sample_idx += 1

        # Calculate mean per dataset
        dataset_performance[model_id] = {}
        for dataset_name, dice_scores in dataset_metrics.items():
            dataset_performance[model_id][dataset_name] = {
                'mean': np.mean(dice_scores),
                'std': np.std(dice_scores),
                'count': len(dice_scores)
            }

    return dataset_performance

print("Analyzing per-dataset performance...\n")
dataset_performance = analyze_per_dataset_performance(
    all_predictions_dict, all_targets_dict, all_datasets_dict, MODEL_CONFIGS
)

# Print per-dataset results
print(f"{'='*70}")
print("PER-DATASET PERFORMANCE".center(70))
print(f"{'='*70}\n")

for model_id, config in MODEL_CONFIGS.items():
    print(f"{config['name']}:")
    for dataset, metrics in dataset_performance[model_id].items():
        print(f"  {dataset}: {metrics['mean']:.4f} ± {metrics['std']:.4f} (n={metrics['count']})")
    print()

# Statistical significance tests (Paired T-Test)
print(f"{'='*70}")
print("STATISTICAL SIGNIFICANCE TESTS".center(70))
print(f"{'='*70}\n")

# Collect per-sample dice scores for each model
model_dice_scores = {}

for model_id in MODEL_CONFIGS.keys():
    predictions = all_predictions_dict[model_id]
    targets = all_targets_dict[model_id]

    dice_scores = []
    for batch_preds, batch_targets in zip(predictions, targets):
        for pred, target in zip(batch_preds, batch_targets):
            pred_tensor = torch.from_numpy(pred)
            target_tensor = torch.from_numpy(target)
            metrics = calculate_metrics(pred_tensor, target_tensor)
            dice_scores.append(metrics['overall']['dice'])

    model_dice_scores[model_id] = np.array(dice_scores)

# Perform paired t-tests
print("Paired T-Tests (comparing models on same samples):\n")

comparisons = [
    ('model1', 'model2'),
    ('model1', 'model3'),
    ('model2', 'model3')
]

for model_a, model_b in comparisons:
    scores_a = model_dice_scores[model_a]
    scores_b = model_dice_scores[model_b]

    # Paired t-test
    t_stat, p_value = stats.ttest_rel(scores_a, scores_b)

    mean_diff = np.mean(scores_b - scores_a)

    name_a = MODEL_CONFIGS[model_a]['name']
    name_b = MODEL_CONFIGS[model_b]['name']

    print(f"{name_b} vs {name_a}:")
    print(f"  Mean difference: {mean_diff:.4f}")
    print(f"  T-statistic: {t_stat:.4f}")
    print(f"  P-value: {p_value:.6f}")

    if p_value < 0.001:
        significance = "*** (p < 0.001) - Highly significant"
    elif p_value < 0.01:
        significance = "** (p < 0.01) - Very significant"
    elif p_value < 0.05:
        significance = "* (p < 0.05) - Significant"
    else:
        significance = "Not significant (p ≥ 0.05)"

    print(f"  Significance: {significance}")

    if mean_diff > 0:
        print(f"  ✓ {name_b} performs better")
    else:
        print(f"  ✓ {name_a} performs better")
    print()

# 95% Confidence Intervals
print(f"{'='*70}")
print("95% CONFIDENCE INTERVALS FOR DICE SCORES".center(70))
print(f"{'='*70}\n")

for model_id, config in MODEL_CONFIGS.items():
    scores = model_dice_scores[model_id]
    mean = np.mean(scores)
    std_err = stats.sem(scores)
    ci = stats.t.interval(0.95, len(scores)-1, loc=mean, scale=std_err)

    print(f"{config['name']}:")
    print(f"  Mean: {mean:.4f}")
    print(f"  95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
    print()

print(f"✅ Phase 4 Complete: Statistical validation done\n")

In [ ]:
# ============================================================================
# PHASE 5: BEST MODEL SELECTION & SAVING
# ============================================================================

print(f"\n{'='*70}")
print("PHASE 5: BEST MODEL SELECTION".center(70))
print(f"{'='*70}\n")

# Scoring system for model selection
def calculate_model_score(model_id, all_results, efficiency_results, weights=None):
    """Calculate weighted score for model selection"""

    if weights is None:
        weights = {
            'dice': 0.5,           # 50% weight on accuracy
            'inference_time': 0.2, # 20% weight on speed (inverse)
            'model_size': 0.2,     # 20% weight on size (inverse)
            'generalization': 0.1  # 10% weight on low std
        }

    # Normalize metrics (0-1 scale)
    dice_score = all_results[model_id]['overall']['dice']['mean']
    dice_std = all_results[model_id]['overall']['dice']['std']
    inference_time = efficiency_results[model_id]['inference_time_ms']
    model_size = efficiency_results[model_id]['model_size_mb']

    # Get min/max for normalization
    all_dice = [all_results[mid]['overall']['dice']['mean'] for mid in MODEL_CONFIGS.keys()]
    all_times = [efficiency_results[mid]['inference_time_ms'] for mid in MODEL_CONFIGS.keys()]
    all_sizes = [efficiency_results[mid]['model_size_mb'] for mid in MODEL_CONFIGS.keys()]
    all_stds = [all_results[mid]['overall']['dice']['std'] for mid in MODEL_CONFIGS.keys()]

    # Normalize (higher is better)
    norm_dice = (dice_score - min(all_dice)) / (max(all_dice) - min(all_dice) + 1e-8)
    norm_time = 1 - (inference_time - min(all_times)) / (max(all_times) - min(all_times) + 1e-8)  # Inverse
    norm_size = 1 - (model_size - min(all_sizes)) / (max(all_sizes) - min(all_sizes) + 1e-8)  # Inverse
    norm_std = 1 - (dice_std - min(all_stds)) / (max(all_stds) - min(all_stds) + 1e-8)  # Inverse

    # Weighted score
    final_score = (
        weights['dice'] * norm_dice +
        weights['inference_time'] * norm_time +
        weights['model_size'] * norm_size +
        weights['generalization'] * norm_std
    )

    return final_score, {
        'dice_contribution': weights['dice'] * norm_dice,
        'speed_contribution': weights['inference_time'] * norm_time,
        'size_contribution': weights['model_size'] * norm_size,
        'generalization_contribution': weights['generalization'] * norm_std
    }

# Calculate scores for all models
model_scores = {}
for model_id in MODEL_CONFIGS.keys():
    score, breakdown = calculate_model_score(model_id, all_results, efficiency_results)
    model_scores[model_id] = {'score': score, 'breakdown': breakdown}

# Print ranking
print("MODEL RANKING (Weighted Score):\n")
ranked_models = sorted(model_scores.items(), key=lambda x: x[1]['score'], reverse=True)

for rank, (model_id, data) in enumerate(ranked_models, 1):
    config = MODEL_CONFIGS[model_id]
    print(f"{rank}. {config['name']}")
    print(f"   Overall Score: {data['score']:.4f}")
    print(f"   Breakdown:")
    print(f"     - Accuracy:       {data['breakdown']['dice_contribution']:.4f}")
    print(f"     - Speed:          {data['breakdown']['speed_contribution']:.4f}")
    print(f"     - Size:           {data['breakdown']['size_contribution']:.4f}")
    print(f"     - Generalization: {data['breakdown']['generalization_contribution']:.4f}")
    print()

# Select best model
best_model_id = ranked_models[0][0]
best_model_config = MODEL_CONFIGS[best_model_id]
best_model = models[best_model_id]

print(f"{'='*70}")
print(f"🏆 BEST MODEL: {best_model_config['name']}")
print(f"{'='*70}")
print(f"Test Dice Score: {all_results[best_model_id]['overall']['dice']['mean']:.4f}")
print(f"Artery Dice: {all_results[best_model_id]['class_1']['dice']['mean']:.4f}")
print(f"Vein Dice: {all_results[best_model_id]['class_2']['dice']['mean']:.4f}")
print(f"Inference Time: {efficiency_results[best_model_id]['inference_time_ms']:.2f} ms")
print(f"Model Size: {efficiency_results[best_model_id]['model_size_mb']:.2f} MB")
print(f"{'='*70}\n")

# Save best model
best_model_dir = f"{OUTPUT_DIR}/best_model"
os.makedirs(best_model_dir, exist_ok=True)

# Save PyTorch model
torch.save({
    'model_state_dict': best_model.state_dict(),
    'config': best_model_config,
    'test_results': all_results[best_model_id],
    'efficiency': efficiency_results[best_model_id],
}, f"{best_model_dir}/best_model.pth")

print(f"✓ Best model saved to: {best_model_dir}/best_model.pth")

# Save model card (metadata)
model_card = {
    'model_name': best_model_config['name'],
    'architecture': best_model_config['architecture'],
    'encoder': best_model_config['encoder'],
    'performance': {
        'test_dice_overall': float(all_results[best_model_id]['overall']['dice']['mean']),
        'test_dice_artery': float(all_results[best_model_id]['class_1']['dice']['mean']),
        'test_dice_vein': float(all_results[best_model_id]['class_2']['dice']['mean']),
        'test_iou_overall': float(all_results[best_model_id]['overall']['iou']['mean']),
        'test_sensitivity': float(all_results[best_model_id]['overall']['sensitivity']['mean']),
        'test_specificity': float(all_results[best_model_id]['overall']['specificity']['mean']),
    },
    'efficiency': {
        'inference_time_ms': float(efficiency_results[best_model_id]['inference_time_ms']),
        'fps': float(efficiency_results[best_model_id]['fps']),
        'model_size_mb': float(efficiency_results[best_model_id]['model_size_mb']),
        'parameters': int(efficiency_results[best_model_id]['total_params']),
    },
    'training': {
        'image_size': 512,
        'num_classes': 3,
        'class_names': ['Background', 'Artery', 'Vein'],
    },
    'limitations': [
        'Performance may degrade on very low-quality images',
        'Trained on DRIVE, STARE, and CHASE_DB1 datasets',
        'May require fine-tuning for other retinal imaging devices',
    ],
    'recommended_use': [
        'Retinal vessel segmentation and analysis',
        'Artery-vein classification in fundus images',
        'Supporting diagnosis of retinal diseases',
    ]
}

with open(f"{best_model_dir}/model_card.json", 'w') as f:
    json.dump(model_card, f, indent=2)

print(f"✓ Model card saved to: {best_model_dir}/model_card.json")

# Export to ONNX (optional but recommended for deployment)
print(f"\nExporting to ONNX format...")
try:
    dummy_input = torch.randn(1, 3, 512, 512).to(device)
    onnx_path = f"{best_model_dir}/best_model.onnx"

    torch.onnx.export(
        best_model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )

    print(f"✓ ONNX model exported to: {onnx_path}")
except Exception as e:
    print(f"⚠️  ONNX export failed: {e}")
    print("   (This is optional - PyTorch model is already saved)")

print(f"\n✅ Phase 5 Complete: Best model selected and saved\n")

In [ ]:
# ============================================================================
# GENERATE FINAL SUMMARY REPORT
# ============================================================================

print(f"\n{'='*70}")
print("GENERATING FINAL SUMMARY REPORT".center(70))
print(f"{'='*70}\n")

summary_report = []

summary_report.append("="*70)
summary_report.append("AI-BASED ARTERY-VEIN SEGMENTATION")
summary_report.append("EVALUATION SUMMARY REPORT")
summary_report.append("="*70)
summary_report.append("")

summary_report.append("1. MODELS EVALUATED")
summary_report.append("-" * 70)
for model_id, config in MODEL_CONFIGS.items():
    summary_report.append(f"   • {config['name']}")
    summary_report.append(f"     Architecture: {config['architecture']}")
    summary_report.append(f"     Encoder: {config['encoder']}")
    summary_report.append(f"     Parameters: {efficiency_results[model_id]['total_params']:,}")
summary_report.append("")

summary_report.append("2. TEST SET PERFORMANCE")
summary_report.append("-" * 70)
for model_id, config in MODEL_CONFIGS.items():
    summary_report.append(f"   {config['name']}:")
    summary_report.append(f"     Overall Dice:  {all_results[model_id]['overall']['dice']['mean']:.4f} ± {all_results[model_id]['overall']['dice']['std']:.4f}")
    summary_report.append(f"     Artery Dice:   {all_results[model_id]['class_1']['dice']['mean']:.4f}")
    summary_report.append(f"     Vein Dice:     {all_results[model_id]['class_2']['dice']['mean']:.4f}")
    summary_report.append(f"     IoU:           {all_results[model_id]['overall']['iou']['mean']:.4f}")
    summary_report.append(f"     Sensitivity:   {all_results[model_id]['overall']['sensitivity']['mean']:.4f}")
    summary_report.append(f"     Specificity:   {all_results[model_id]['overall']['specificity']['mean']:.4f}")
    summary_report.append("")

summary_report.append("3. EFFICIENCY METRICS")
summary_report.append("-" * 70)
for model_id, config in MODEL_CONFIGS.items():
    summary_report.append(f"   {config['name']}:")
    summary_report.append(f"     Inference Time: {efficiency_results[model_id]['inference_time_ms']:.2f} ms")
    summary_report.append(f"     FPS:            {efficiency_results[model_id]['fps']:.2f}")
    summary_report.append(f"     Model Size:     {efficiency_results[model_id]['model_size_mb']:.2f} MB")
    summary_report.append("")

summary_report.append("4. BEST MODEL SELECTION")
summary_report.append("-" * 70)
summary_report.append(f"   🏆 Winner: {best_model_config['name']}")
summary_report.append(f"   Selected based on weighted scoring:")
summary_report.append(f"     - Accuracy: 50%")
summary_report.append(f"     - Speed: 20%")
summary_report.append(f"     - Size: 20%")
summary_report.append(f"     - Generalization: 10%")
summary_report.append("")

summary_report.append("5. KEY FINDINGS")
summary_report.append("-" * 70)

# Determine key findings based on results
best_dice_model = max(MODEL_CONFIGS.keys(), key=lambda x: all_results[x]['overall']['dice']['mean'])
fastest_model = min(MODEL_CONFIGS.keys(), key=lambda x: efficiency_results[x]['inference_time_ms'])
smallest_model = min(MODEL_CONFIGS.keys(), key=lambda x: efficiency_results[x]['model_size_mb'])

summary_report.append(f"   • Best Accuracy: {MODEL_CONFIGS[best_dice_model]['name']}")
summary_report.append(f"   • Fastest Inference: {MODEL_CONFIGS[fastest_model]['name']}")
summary_report.append(f"   • Smallest Model: {MODEL_CONFIGS[smallest_model]['name']}")
summary_report.append("")

summary_report.append("6. STATISTICAL SIGNIFICANCE")
summary_report.append("-" * 70)
summary_report.append("   Paired t-tests were conducted to compare models.")
summary_report.append("   See statistical validation section for detailed p-values.")
summary_report.append("")

summary_report.append("7. OUTPUT FILES GENERATED")
summary_report.append("-" * 70)
summary_report.append(f"   • Model comparison tables: {OUTPUT_DIR}/tables/")
summary_report.append(f"   • Performance visualizations: {OUTPUT_DIR}/figures/")
summary_report.append(f"   • Sample predictions: {OUTPUT_DIR}/predictions/")
summary_report.append(f"   • Best model saved: {OUTPUT_DIR}/best_model/")
summary_report.append("")

summary_report.append("8. RECOMMENDATIONS")
summary_report.append("-" * 70)
summary_report.append(f"   For deployment: Use {best_model_config['name']}")
summary_report.append(f"   Reason: Best balance of accuracy and efficiency")
summary_report.append("")
summary_report.append("   For maximum accuracy: Use " + MODEL_CONFIGS[best_dice_model]['name'])
summary_report.append("   For fastest inference: Use " + MODEL_CONFIGS[fastest_model]['name'])
summary_report.append("")

summary_report.append("="*70)
summary_report.append("EVALUATION COMPLETE")
summary_report.append("="*70)

# Print summary
summary_text = "\n".join(summary_report)
print(summary_text)

# Save summary to file
with open(f"{OUTPUT_DIR}/EVALUATION_SUMMARY.txt", 'w') as f:
    f.write(summary_text)

print(f"\n✅ Summary report saved to: {OUTPUT_DIR}/EVALUATION_SUMMARY.txt\n")